<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 0 · Quickstart

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/00_dxquant_quickstart.ipynb)


## Notebook Goals

This quickstart consolidates the most common DXQuant workflows:

- simulate risk factor paths under Geometric Brownian Motion (GBM),
- price European, American, and path-dependent options,
- illustrate SABR implied-volatility smiles, and
- aggregate simple portfolios and compute distribution-based risk metrics.


Sampling errors use independent paths or antithetic pair means. A single moment-matched batch reports `NaN` for its error; use independent seeded replications to measure batch dispersion. A shared `scenario_set` explicitly asserts ordered joint scenarios for portfolio payoff risk; it does not construct correlations.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt
from pprint import pprint

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

ENABLE_PLOTS = False

plt = None
if ENABLE_PLOTS:
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError:
        plt = None

## 1. Model Setup

We use a constant short rate for discounting and a GBM for the underlying dynamics.

In [ ]:
pricing_date = dt.date(2024, 1, 1)
maturity = dt.date(2025, 1, 1)

spot = 100.0
strike = 105.0
rate = 0.02
vol = 0.20

env = dx.MarketEnvironment(name="qs", pricing_date=pricing_date)
env.add_curve("discount", dx.ConstantShortRateCurve("disc", pricing_date, rate))

process = dx.GeometricBrownianMotion(
    drift=rate,
    volatility=vol,
    seed=123,
    moment_matching=False,
)
pricer = dx.MonteCarloPricer(
    env=env,
    process=process,
    discount_curve_key="discount",
    paths=40000,
    steps=96,
)

pprint(env.snapshot())

## 2. Path Simulation

A GBM generates a panel of simulated prices sampled on a uniform time grid.

In [ ]:
maturity_years = float(
    dx.year_fractions([pricing_date, maturity], day_count=365.0)[1]
)
grid = dx.uniform_time_grid(maturity_years, pricer.steps)
paths = process.simulate_paths(spot=spot, time_grid=grid, paths=5000)

print(paths.shape)
print("terminal mean:", float(np.mean(paths[:, -1])))
print("terminal std:", float(np.std(paths[:, -1], ddof=1)))

if plt is not None:
    plt.figure(figsize=(10, 4))
    plt.plot(paths[:25].T, lw=1.0, alpha=0.7)
    plt.title("GBM sample paths")
    plt.xlabel("time step")
    plt.ylabel("price")
    plt.show()

## 3. European Call: Monte Carlo and Analytical Benchmark

The European call is priced by Monte Carlo, with an analytical Black–Scholes benchmark via
`dxquant.valuation.EuropeanOptionValuation`.

In [ ]:
call = dx.EuropeanCallPayoff(strike=strike)
valuation = dx.EuropeanOptionValuation(
    pricer=pricer,
    payoff=call,
    maturity=maturity,
    spot=spot,
)

mc = pricer.price_terminal(call, maturity=maturity, spot=spot)
analytical = valuation.analytical_price()

print(f"Monte Carlo price:   {mc.price:.6f}")
print(f"Std. error:          {mc.std_error:.6f}")
print(f"Analytical price:    {float(analytical):.6f}")
print(f"Abs. difference:     {abs(mc.price - float(analytical)):.6f}")

## 4. American Put: Longstaff–Schwartz

DXQuant provides a compact Longstaff–Schwartz implementation for a single-asset American put.

In [ ]:
disc = env.get_curve("discount")
lsm = dx.LongstaffSchwartzPricer(
    process=process,
    discount_curve=disc,
    paths=40000,
    steps=50,
)

american_put = lsm.price_american_put(
    spot=spot,
    strike=strike,
    maturity_years=maturity_years,
)

print(f"American put (LSM):  {american_put:.6f}")

## 5. Path-Dependent Options

Path-dependent payoffs (Asian, barrier, baskets, ...) evaluate the full simulated path.

In [ ]:
asian = dx.ArithmeticAsianCallPayoff(strike=strike)
asian_result = pricer.price_path_dependent(asian, maturity=maturity, spot=spot)

barrier = dx.UpAndOutBarrierCallPayoff(strike=strike, barrier=130.0)
barrier_result = pricer.price_path_dependent(
    barrier, maturity=maturity, spot=spot
)

print(f"Asian call price:    {asian_result.price:.6f}")
print(f"Barrier call price:  {barrier_result.price:.6f}")

## 6. SABR Smile (Implied Volatility)

DXQuant includes Hagan-style SABR implied volatility as a lightweight analytical building block.

In [ ]:
forward = 0.05
alpha = 0.10
beta = 0.50
rho = 0.10
vol_vol = 0.50

sabr_strikes = np.linspace(0.02, 0.10, 9)
sabr_vols = np.array(
    [
        dx.sabr_implied_volatility(
            forward=forward,
            strike=float(k),
            maturity=maturity_years,
            alpha=alpha,
            beta=beta,
            rho=rho,
            vol_vol=vol_vol,
        )
        for k in sabr_strikes
    ],
    dtype=float,
)

for k, v in zip(sabr_strikes, sabr_vols, strict=True):
    print(f"strike={k:.4f}  vol={v:.4f}")

if plt is not None:
    plt.figure(figsize=(8, 4))
    plt.plot(sabr_strikes, sabr_vols, marker="o")
    plt.title("SABR implied volatility smile")
    plt.xlabel("strike")
    plt.ylabel("implied vol")
    plt.show()

## 7. Simple Portfolio Aggregation

Portfolio aggregation uses the discounted payoff distributions. Here all positions share the
same pricer and underlying, so combining path-wise discounted payoffs is meaningful.

In [ ]:
portfolio_pricer = dx.PortfolioPricer()

specs = [
    dx.PositionSpec(
        position_id="call",
        pricer=pricer,
        payoff=call,
        maturity=maturity,
        scenario_set="shared-gbm-seed-123",
        spot=spot,
        quantity=3.0,
    ),
    dx.PositionSpec(
        position_id="barrier",
        pricer=pricer,
        payoff=barrier,
        maturity=maturity,
        scenario_set="shared-gbm-seed-123",
        spot=spot,
        quantity=2.0,
        path_dependent=True,
    ),
]

port = portfolio_pricer.price(specs)

print(f"Portfolio price:     {port.price:.6f}")
print(f"Portfolio VaR 95%:   {port.value_at_risk(0.95):.6f}")
print(f"Portfolio ES 95%:    {port.expected_shortfall(0.95):.6f}")

## 8. Scenario Analysis

A scenario modifies position specifications (for example scaling a spot) and reprices the
portfolio.

In [ ]:
analyzer = dx.ScenarioAnalyzer(portfolio_pricer, specs)
scenarios = [
    dx.Scenario(name="base", modifier=lambda s: s),
    dx.Scenario(name="spot +5%", modifier=dx.scale_spot("call", 1.05)),
    dx.Scenario(name="spot -5%", modifier=dx.scale_spot("call", 0.95)),
]

results = analyzer.run(scenarios)
for res in results:
    print(f"{res.name:>9s}: {res.result.price: .6f}")

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">